## Chapter 4: Getting to Know Your Data/Project:
* Retrieval Augmented Generation Chatbot/CH4-01-Creating VectorDB.py

In [0]:
%pip install -q transformers "unstructured[pdf,docx]" langchain llama-index databricks-vectorsearch pydantic mlflow
%restart_python

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc_endpoint_name = "one-env-shared-endpoint-1"#"ml_action_vs"
vsc = VectorSearchClient()

# SIDE NOTE - experience strange behaviour with the Index re-provisioning 
if vsc_endpoint_name not in [e['name'] for e in vsc.list_endpoints().get('endpoints', [])]:
  vsc.create_endpoint(name=vsc_endpoint_name, endpoint_type="STANDARD")
  print(f"Endpoint named {vsc_endpoint_name} is in creation, wait a moment.")
 
print(f"Endpoint named {vsc_endpoint_name} is ready.")

In [0]:
display(spark.read.table("porya_catalog.default.pdf_documentation_text"))


In [0]:
#from utils 
def index_exists(vsc, endpoint_name, index_full_name):
  try:
      dict_vsindex = vsc.get_index(endpoint_name, index_full_name).describe()
      return dict_vsindex.get('status').get('ready', False)
  except Exception as e:
      if 'RESOURCE_DOES_NOT_EXIST' not in str(e):
          print(f'Unexpected error describing the index. This could be a permission issue.')
          raise e
  return False

In [0]:
from databricks.sdk import WorkspaceClient
#import databricks.sdk.service.catalog as c

#The table we'd like to index
source_table_fullname = f"porya_catalog.default.pdf_documentation_text"
# Where we want to store our index
vs_index_fullname = "porya_catalog.default.docs_vsc_idx_cont"

try:
  idx = vsc.get_index(vsc_endpoint_name, vs_index_fullname)
  idx_status = idx.describe().get('status', {})
  if idx_status.get('ready', False):
    print(f"Index {vs_index_fullname} is ready, triggering sync...")
    idx.sync()
  else:
    print(f"Index {vs_index_fullname} exists but is not ready yet.")
    print(f"Status: {idx_status.get('detailed_state', 'unknown')} - {idx_status.get('message', '')}")
except Exception as e:
  if 'does not exist' in str(e).lower() or 'RESOURCE_DOES_NOT_EXIST' in str(e):
    print(f"Creating index {vs_index_fullname} on endpoint {vsc_endpoint_name}...")
    vsc.create_delta_sync_index(
      endpoint_name=vsc_endpoint_name,
      index_name=vs_index_fullname,
      source_table_name=source_table_fullname,
      pipeline_type="TRIGGERED", # TRIGGERED or CONTINUOUS 
      primary_key="id",
      embedding_dimension=1024, #Match your model embedding size (gte)
      embedding_vector_column="embedding"
    )
  else:
    raise e

In [0]:
def wait_for_index_to_be_ready(vsc, vs_endpoint_name, index_name):
  for i in range(180):
    idx = vsc.get_index(vs_endpoint_name, index_name).describe()
    index_status = idx.get('status', idx.get('index_status', {}))
    status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
    url = index_status.get('index_url', index_status.get('url', 'UNKNOWN'))
    if "ONLINE" in status:
      return
    if "UNKNOWN" in status:
      print(f"Can't get the status - will assume index is ready {idx} - url: {url}")
      return
    elif "PROVISIONING" in status:
      if i % 40 == 0: print(f"Waiting for index to be ready, this can take a few min... {index_status} - pipeline url:{url}")
      time.sleep(10)
    else:
        raise Exception(f'''Error with the index - this shouldn't happen. DLT pipeline might have been killed.\n Please delete it and re-run the previous cell: vsc.delete_index("{index_name}, {vs_endpoint_name}") \nIndex details: {idx}''')
  raise Exception(f"Timeout, your index isn't ready yet: {vsc.get_index(index_name, vs_endpoint_name)}")

In [0]:
#Let's wait for the index to be ready and all our embeddings to be created and indexed
import time

wait_for_index_to_be_ready(vsc, vsc_endpoint_name, vs_index_fullname)

In [0]:
# Checking the information about the VS 
vsc.get_index(vsc_endpoint_name, vs_index_fullname).describe()

In [0]:
def pprint(obj):
  import pprint
  pprint.pprint(obj, compact=True, indent=1, width=100)

In [0]:
import mlflow.deployments
deploy_client = mlflow.deployments.get_deploy_client("databricks")

question = "The Economic Impacts of Automation Technologies using LLMs"

response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [question]})
embeddings = [e['embedding'] for e in response.data]

results = vsc.get_index(vsc_endpoint_name, vs_index_fullname).similarity_search(
  query_vector=embeddings[0],
  columns=["pdf_name", "content"],
  num_results=3)
docs = results.get('result', {}).get('data_array', [])
pprint(docs)